In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
train_df = pd.read_csv("data/booking_reviews_train.csv")
test_df = pd.read_csv("data/booking_reviews_test.csv")

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train = vectorizer.fit_transform(train_df["text_combined"])
X_test = vectorizer.transform(test_df["text_combined"])

y_train = train_df["is_positive"].values
y_test = test_df["is_positive"].values

X_train = X_train.toarray()
X_test = X_test.toarray()

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [3]:
class SentimentNN(nn.Module):
    def __init__(self, input_dim):
        super(SentimentNN, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [4]:
input_dim = X_train.shape[1]

model = SentimentNN(input_dim)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
batch_size = 64

for epoch in range(epochs):
    permutation = torch.randperm(X_train.size(0))
    
    epoch_loss = 0
    
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_X = X_train[indices]
        batch_y = y_train[indices]
        
        optimizer.zero_grad()
        
        outputs = model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

Epoch 1, Loss: 129.6187
Epoch 2, Loss: 78.0637
Epoch 3, Loss: 59.0246
Epoch 4, Loss: 44.1530
Epoch 5, Loss: 34.4209
Epoch 6, Loss: 28.5122
Epoch 7, Loss: 25.9850
Epoch 8, Loss: 24.0681
Epoch 9, Loss: 23.0857
Epoch 10, Loss: 22.9016


In [5]:
outputs = model(X_test)
preds = (outputs >= 0.5).int()

with torch.no_grad():
    outputs = model(X_test).squeeze()
    preds = (outputs >= 0.5).int().numpy()

y_true = y_test.numpy()

print("Accuracy:", accuracy_score(y_true, preds))
print("Precision:", precision_score(y_true, preds))
print("Recall:", recall_score(y_true, preds))
print("F1 Score:", f1_score(y_true, preds))

Accuracy: 0.8242908814011042
Precision: 0.9125421822272216
Recall: 0.8413274565724657
F1 Score: 0.8754890058006205
